# 03 · Multiple months — trends the direct way

Every notebook so far has looked at **one month**. This one stacks months into a time series, exactly the same way it loads one month — in a loop.

> **Two honest notes before the charts.**
> 1. **This is the simple path, on purpose.** Each month downloads directly from the manifest, like notebooks 00–02. For a handful of months that is all you need. If you ever need dozens of months repeatedly, a local cache is the optimisation to reach for — not needed here.
> 2. **The data are not cleaned.** Treat every line below as exploratory — great for questions, not for publishing claims yet.

## Setup — the same loader as notebooks 00–02

In [ ]:
# ── Setup: imports, manifest, catalog ─────────────────────────────────────────
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

MANIFEST_URL = "https://measurementlab.net/data/stats/manifest.json"
manifest = requests.get(MANIFEST_URL, timeout=30).json()

records = []
for path, meta in manifest["files"].items():
    parts = path.split("/")
    # cache/v1/{start_ts}/{end_ts}/{slice_name}/data.parquet
    if len(parts) == 6 and parts[5] == "data.parquet":
        records.append({
            "start": pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "end":   pd.to_datetime(parts[3], format="%Y%m%dT%H%M%SZ"),
            "slice": parts[4],
            "url":   meta["url"],
        })

catalog = (pd.DataFrame(records)
           .sort_values(["slice", "start"])
           .reset_index(drop=True))


def month_url(slice_name, start):
    # Direct URL of one month of one slice. start: 'YYYY-MM-DD' (first of month).
    row = catalog[(catalog["slice"] == slice_name) &
                  (catalog["start"] == pd.to_datetime(start))]
    if row.empty:
        raise ValueError(f"No {slice_name} file for {start}")
    return row.iloc[0]["url"]


print("Catalog loaded —", len(catalog), "files,",
      catalog["slice"].nunique(), "slices.")

## Load the last 12 months for a few countries

A loop over months, each `pd.read_parquet` on the direct URL, one `concat` — that is the whole mechanism. The month range comes from the manifest, never typed by hand.

In [ ]:
N_MONTHS = 12
COUNTRIES = ["US", "DE", "BR", "IN", "NG"]

months = sorted(catalog.loc[catalog["slice"] == "downloads_by_country", "start"]
                .dt.strftime("%Y-%m-%d").unique())[-N_MONTHS:]

frames = []
for m in months:
    dl = pd.read_parquet(month_url("downloads_by_country", m))
    ul = pd.read_parquet(month_url("uploads_by_country", m))
    merged = dl.merge(ul[["country_code", "upload_p50"]], on="country_code", how="left")
    merged = merged[merged["country_code"].isin(COUNTRIES)].copy()
    merged["month"] = pd.to_datetime(m)
    frames.append(merged)

ts = pd.concat(frames, ignore_index=True).sort_values("month")
print("Loaded", len(months), "months ×", ts["country_code"].nunique(), "countries.")
print("Months:", ", ".join(m[:7] for m in months))

## The four metrics over time

Four panels — download, upload, latency, loss — for the selected countries. Median per month: the typical user's experience, month by month.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
panels = [
    ("download_p50", "Median download (Mbit/s)", axes[0, 0]),
    ("upload_p50",   "Median upload (Mbit/s)",   axes[0, 1]),
    ("latency_p50",  "Median latency (ms) — lower is better", axes[1, 0]),
    ("loss_p50",     "Median packet loss — lower is better",  axes[1, 1]),
]

for col, ylabel, ax in panels:
    for cc, grp in ts.groupby("country_code"):
        grp = grp.sort_values("month")
        ax.plot(grp["month"], grp[col], marker="o", markersize=4, label=cc)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

plt.suptitle(f"Country medians, {ts['month'].min().date()} → {ts['month'].max().date()}",
             fontsize=13)
plt.tight_layout(); plt.show()

print("Reading it aloud: pick one country and one panel — is the line rising, "
      "falling, or flat? That is a trend question this dataset can answer.")

## Make it yours

Change `COUNTRIES` or `N_MONTHS` in the cell above, then re-run the last two cells. The charts redraw for your country, your slice of time.

**Check yourself.** We built the median line only. Why not the p25–p75 band? *Because on un-cleaned data a band can look like a real distribution shift when it is just noise — so this workshop keeps the exploratory view honest: medians, and the sample counts behind them.*